# 01 — Data analysis and MRI modality comparison

Statistics per modality, histograms, threshold baseline + ablation over the 15 modality combinations,
tumor localisation and patient clustering.

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
from braintumorlab import analysis as A
from braintumorlab.volumes import list_patients, load_patient
SPLIT = "train"
pid, split, folder = list_patients(DATA_ROOT, SPLIT)[0]
p = load_patient(folder)          # background set to 0, z-score inside the brain
print(pid, "| brain voxels:", p["brain"].sum(), "| tumor voxels:", (p["seg"] > 0).sum())

In [ ]:
A.plot_modalities_with_seg(p)     # labels: 1 = NCR/NET (red), 2 = ED (green), 4 = ET (yellow)

In [ ]:
A.plot_histograms(p)

### Statistics per modality
CNR = (mean tumor - mean healthy) / std healthy. The plain SNR (mean / std) is meaningless on z-scored data
(the mean is about 0), so the CNR is the number to compare between modalities.

In [ ]:
A.modality_table(p).round(3)

### Threshold baseline + ablation (15 combinations)
Each modality is oriented so that the tumor is "higher" (the tumor is hypo-intense in T1). This is a
simple **baseline**, not an AI model: it measures how much tumor information each sequence carries.

In [ ]:
abl = A.ablation_study(p, percentile=90)
abl.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, ["f1", "recall", "auc"]):
    ax.barh(abl["scenario"], abl[m]); ax.set_xlim(0, 1); ax.set_title(m); ax.invert_yaxis()
plt.tight_layout(); plt.show()
print("Best combination:", abl.iloc[0]["scenario"], "| weakest:", abl.iloc[-1]["scenario"])

### Ablation over several patients (more reliable than one patient)

In [ ]:
import pandas as pd
rows = []
for pid_, _, f in list_patients(DATA_ROOT, SPLIT, max_patients=20):
    r = A.ablation_study(load_patient(f)); r["patient_id"] = pid_; rows.append(r)
abl_all = pd.concat(rows).groupby("scenario")[["f1", "recall", "auc"]].agg(["mean", "std"]).sort_values(("f1", "mean"), ascending=False)
abl_all.round(3)

### Tumor localisation (uses the NIfTI orientation)

In [ ]:
A.tumor_localization(p['seg'], p['brain'], p['affine'])

### Patient clustering
K-Means runs on the standardised features; UMAP (or PCA) is only used to draw the 2D map.

In [ ]:
!pip install -q umap-learn
FEATURE_CASE = "all_modalities"      # see A.FEATURE_CASES
feats = A.features_table(DATA_ROOT, "train", max_patients=60, case=FEATURE_CASE)
clusters, emb = A.cluster_patients(feats, n_clusters=3)
plt.figure(figsize=(7, 6)); plt.scatter(emb[:, 0], emb[:, 1], c=clusters["cluster"], cmap="tab10")
plt.title(f"Patients - {FEATURE_CASE}"); plt.show()
clusters.groupby("cluster").mean(numeric_only=True).round(3)

### Generation metrics
SSIM / PSNR / MAE are computed on images generated by Med-DDPM in notebook 04
(helper: `A.generation_metrics(real, generated, brain_mask)`).